In [ ]:
# Cell 0 - environment and paths (works in Colab and locally)
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/incident-risk-predictor-files'
    !pip install -q xgboost imbalanced-learn shap
else:
    # Drive for desktop path; override with the IRP_BASE environment variable if yours differs
    BASE = os.environ.get('IRP_BASE', r'G:\My Drive\incident-risk-predictor-files')

for sub in ['datasets/raw', 'datasets/processed', 'model_checkpoints', 'reports', 'logs']:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)
print('IN_COLAB =', IN_COLAB, '| BASE =', BASE)

In [ ]:
from google.colab import files

print("Dataset")
uploaded = files.upload()

# Move them to Drive
import shutil
for filename in uploaded.keys():
    dest = f'{BASE}/datasets/raw/{filename}'
    shutil.copy(filename, dest)
    print(f'Saved to Drive: {dest}')

In [ ]:
from google.colab import files

print("Dataset")
uploaded = files.upload()

# Move them to Drive
import shutil
for filename in uploaded.keys():
    dest = f'{BASE}/datasets/raw/{filename}'
    shutil.copy(filename, dest)
    print(f'Saved to Drive: {dest}')

In [ ]:
import pandas as pd

base = f'{BASE}/datasets/raw'

# Replace with your actual filenames
df1 = pd.read_csv(f'{base}/predictive_maintenance.csv')
df2 = pd.read_csv(f'{base}/incident_event_log.csv')

print("=== Dataset 1 columns ===")
print(df1.columns.tolist())
print(f"Shape: {df1.shape}")

print("\n=== Dataset 2 columns ===")
print(df2.columns.tolist())
print(f"Shape: {df2.shape}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

base = f'{BASE}/datasets/raw'

df1 = pd.read_csv(f'{base}/predictive_maintenance.csv')
df2 = pd.read_csv(f'{base}/incident_event_log.csv')

print("=== Dataset 1: Failure Type distribution ===")
print(df1['Failure Type'].value_counts())
print()
print("=== Dataset 1: Target distribution ===")
print(df1['Target'].value_counts())
print()
print("=== Dataset 1: Type (machine grade) ===")
print(df1['Type'].value_counts())

In [ ]:
print("=== Dataset 2: priority distribution ===")
print(df2['priority'].value_counts())
print()
print("=== Dataset 2: impact distribution ===")
print(df2['impact'].value_counts())
print()
print("=== Dataset 2: urgency distribution ===")
print(df2['urgency'].value_counts())
print()
print("=== Dataset 2: incident_state distribution ===")
print(df2['incident_state'].value_counts())
print()
print("=== Dataset 2: category distribution ===")
print(df2['category'].value_counts().head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

base = f'{BASE}/datasets/raw'
processed_base = f'{BASE}/datasets/processed'

# ── Load ──────────────────────────────────────────────────────────────────
df1 = pd.read_csv(f'{base}/predictive_maintenance.csv')
df2 = pd.read_csv(f'{base}/incident_event_log.csv')

# ══════════════════════════════════════════════════════════════════════════
# DATASET 1 — Predictive Maintenance
# ══════════════════════════════════════════════════════════════════════════
def map_risk_d1(failure_type):
    if failure_type == 'No Failure':
        return 'Low'
    elif failure_type in ['Heat Dissipation Failure', 'Power Failure']:
        return 'Medium'
    else:  # Overstrain, Tool Wear, Random
        return 'High'

df1['risk_level'] = df1['Failure Type'].apply(map_risk_d1)

# Select & rename features to match our schema
d1_clean = pd.DataFrame({
    'incident_type':      df1['Failure Type'],
    'department':         df1['Type'].map({'L': 'Line_Maintenance', 'M': 'Heavy_Maintenance', 'H': 'Engineering'}),
    'shift_type':         np.random.choice(['Morning', 'Afternoon', 'Night'], size=len(df1)),
    'severity_score':     df1['Torque [Nm]'] / df1['Torque [Nm]'].max(),
    'tool_wear_min':      df1['Tool wear [min]'],
    'air_temp_k':         df1['Air temperature [K]'],
    'process_temp_k':     df1['Process temperature [K]'],
    'rotational_speed':   df1['Rotational speed [rpm]'],
    'operational_impact': df1['Target'].map({0: 'Low', 1: 'High'}),
    'task_status':        np.random.choice(['Complete', 'In Progress', 'Pending'], size=len(df1), p=[0.6, 0.3, 0.1]),
    'resolution_time':    np.random.randint(1, 480, size=len(df1)),  # minutes
    'source':             'maintenance',
    'risk_level':         df1['risk_level']
})

print(f"Dataset 1 cleaned: {d1_clean.shape}")
print(d1_clean['risk_level'].value_counts())

# ══════════════════════════════════════════════════════════════════════════
# DATASET 2 — Incident Event Log (sample 5000 for balance)
# ══════════════════════════════════════════════════════════════════════════
def map_risk_d2(priority):
    if str(priority).startswith('4'):
        return 'Low'
    elif str(priority).startswith('3'):
        return 'Medium'
    else:  # 1-Critical, 2-High
        return 'High'

df2['risk_level'] = df2['priority'].apply(map_risk_d2)

# Sample proportionally to keep class distribution
df2_sample = df2.groupby('risk_level', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 1700), random_state=42)
).reset_index(drop=True)

def map_impact(val):
    if '1' in str(val): return 1.0
    elif '2' in str(val): return 0.6
    else: return 0.3

def map_urgency(val):
    if '1' in str(val): return 1.0
    elif '2' in str(val): return 0.6
    else: return 0.3

def map_state(val):
    mapping = {
        'Resolved': 'Complete', 'Closed': 'Complete',
        'Active': 'In Progress', 'New': 'Pending',
        'Awaiting User Info': 'Pending', 'Awaiting Vendor': 'Pending',
        'Awaiting Problem': 'Pending', 'Awaiting Evidence': 'Pending'
    }
    return mapping.get(str(val), 'Pending')

d2_clean = pd.DataFrame({
    'incident_type':      df2_sample['category'].fillna('Unknown'),
    'department':         df2_sample['assignment_group'].fillna('Unknown').astype(str).str[:20],
    'shift_type':         np.random.choice(['Morning', 'Afternoon', 'Night'], size=len(df2_sample)),
    'severity_score':     df2_sample['impact'].apply(map_impact),
    'tool_wear_min':      np.random.randint(0, 300, size=len(df2_sample)),
    'air_temp_k':         np.random.normal(300, 5, size=len(df2_sample)),
    'process_temp_k':     np.random.normal(310, 5, size=len(df2_sample)),
    'rotational_speed':   np.random.randint(1000, 3000, size=len(df2_sample)),
    'operational_impact': df2_sample['impact'].apply(lambda x: 'High' if '1' in str(x) else 'Low'),
    'task_status':        df2_sample['incident_state'].apply(map_state),
    'resolution_time':    df2_sample['reassignment_count'].fillna(0) * 60 + np.random.randint(30, 300, size=len(df2_sample)),
    'source':             'incident_log',
    'risk_level':         df2_sample['risk_level']
})

print(f"\nDataset 2 cleaned: {d2_clean.shape}")
print(d2_clean['risk_level'].value_counts())

In [ ]:
# ── Merge both datasets ───────────────────────────────────────────────────
df_merged = pd.concat([d1_clean, d2_clean], ignore_index=True)
df_merged = df_merged.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"\n=== MERGED DATASET ===")
print(f"Shape: {df_merged.shape}")
print(f"\nRisk Level Distribution:")
print(df_merged['risk_level'].value_counts())
print(f"\nMissing values: {df_merged.isnull().sum().sum()}")
print(f"\nData types:")
print(df_merged.dtypes)

# ── Save processed dataset ────────────────────────────────────────────────
df_merged.to_csv(f'{processed_base}/merged_dataset.csv', index=False)
print(f"\n Saved to Drive: {processed_base}/merged_dataset.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Risk Level Distribution — Incident Risk Prediction Dataset', fontsize=13, fontweight='bold')

colors = {'Low': '#217346', 'Medium': '#FF8C00', 'High': '#C00000'}

# Overall
counts = df_merged['risk_level'].value_counts()
axes[0].bar(counts.index, counts.values, color=[colors[k] for k in counts.index])
axes[0].set_title('Overall (Merged)')
axes[0].set_ylabel('Count')

# By source
for i, (src, ax) in enumerate(zip(['maintenance', 'incident_log'], axes[1:])):
    sub = df_merged[df_merged['source'] == src]['risk_level'].value_counts()
    ax.bar(sub.index, sub.values, color=[colors[k] for k in sub.index])
    ax.set_title(f'Source: {src}')

plt.tight_layout()
plt.savefig(f'{BASE}/reports/merged_distribution.png', dpi=150)
plt.show()
print(" Plot saved to Drive")